# The Scenario: 
You are given three separate datasets: "User Demographics", "Track Metadata" (artist, genre, duration), and "Listening History" (who listened to what, and when).

In [1]:
import pandas as pd

df1 = pd.read_csv('user_demographics.csv')
df2 = pd.read_csv('track_metadata.csv')
df3 = pd.read_csv('listening_history_q1.csv')
df4 = pd.read_csv('listening_history_q2.csv')

print(df2.head())
print(df3.head())

   track_id artist_name       Genre  length_in_ms  internal_db_id
0       100    Artist_0         Pop        203042          645977
1       101    Artist_1         Pop        205999          897606
2       102    Artist_2  Electronic        137955          450605
3       103    Artist_3     Hip-Hop        157841          989465
4       104    Artist_4     Hip-Hop        190640          633636
   listen_id  usr_ref_id  trk_ref_id            timestamp
0          1         167         146  2025-01-23 22:19:45
1          3          90         105  2025-01-06 15:32:21
2          5          39         119  2025-02-12 11:12:04
3          6         126         149  2025-03-05 19:52:42
4          8         173         101   2025-01-04 5:56:14


### Combining Datasets: 
You must use pd.concat() to stitch together listening history from Q1 and Q2. 

In [2]:
df_combined = pd.concat([df3, df4], ignore_index=True)

print(df_combined.head())

   listen_id  usr_ref_id  trk_ref_id            timestamp
0          1         167         146  2025-01-23 22:19:45
1          3          90         105  2025-01-06 15:32:21
2          5          39         119  2025-02-12 11:12:04
3          6         126         149  2025-03-05 19:52:42
4          8         173         101   2025-01-04 5:56:14


Then, use pd.merge() to join the 'Listening History' table with the 'Track Metadata' table so you know the genre of the song played, acting like a SQL left join.

In [3]:
df_merge = pd.merge(df_combined, df2, left_on='trk_ref_id', right_on='track_id', how='left')

print(df_merge.head(5))

   listen_id  usr_ref_id  trk_ref_id            timestamp  track_id  \
0          1         167         146  2025-01-23 22:19:45       146   
1          3          90         105  2025-01-06 15:32:21       105   
2          5          39         119  2025-02-12 11:12:04       119   
3          6         126         149  2025-03-05 19:52:42       149   
4          8         173         101   2025-01-04 5:56:14       101   

  artist_name       Genre  length_in_ms  internal_db_id  
0   Artist_46  Electronic        121062          119870  
1    Artist_5  Electronic        285983          138102  
2   Artist_19     Hip-Hop        134397          625830  
3   Artist_49         Pop        272617          334677  
4    Artist_1         Pop        205999          897606  


### Modifying DataFrames: 
You create a new calculated column called Minutes_Played by dividing the Milliseconds column by 60,000. 

In [4]:
df_merge['Minutes_Played'] = df_merge['length_in_ms'] / 60000

print(df_merge.head())

   listen_id  usr_ref_id  trk_ref_id            timestamp  track_id  \
0          1         167         146  2025-01-23 22:19:45       146   
1          3          90         105  2025-01-06 15:32:21       105   
2          5          39         119  2025-02-12 11:12:04       119   
3          6         126         149  2025-03-05 19:52:42       149   
4          8         173         101   2025-01-04 5:56:14       101   

  artist_name       Genre  length_in_ms  internal_db_id  Minutes_Played  
0   Artist_46  Electronic        121062          119870        2.017700  
1    Artist_5  Electronic        285983          138102        4.766383  
2   Artist_19     Hip-Hop        134397          625830        2.239950  
3   Artist_49         Pop        272617          334677        4.543617  
4    Artist_1         Pop        205999          897606        3.433317  


Use .drop() to clear redundant system ID columns 

In [5]:
cols_to_drop = ['trk_ref_id', 'internal_db_id']
df_merge = df_merge.drop(columns=cols_to_drop)

print(df_merge.head())

   listen_id  usr_ref_id            timestamp  track_id artist_name  \
0          1         167  2025-01-23 22:19:45       146   Artist_46   
1          3          90  2025-01-06 15:32:21       105    Artist_5   
2          5          39  2025-02-12 11:12:04       119   Artist_19   
3          6         126  2025-03-05 19:52:42       149   Artist_49   
4          8         173   2025-01-04 5:56:14       101    Artist_1   

        Genre  length_in_ms  Minutes_Played  
0  Electronic        121062        2.017700  
1  Electronic        285983        4.766383  
2     Hip-Hop        134397        2.239950  
3         Pop        272617        4.543617  
4         Pop        205999        3.433317  


and .rename() poorly named columns.

In [6]:
df_merge = df_merge.rename(columns={
    'listen_id': 'Listen_ID',
    'timestamp': 'Timestamp',
    'track_id': 'Track_ID',
    'usr_ref_id': 'User_ID',
    'artist_name': 'Artist',
    'length_in_ms': 'Duration_ms'
})

df_merge = df_merge.round(2)

print(df_merge.head())

   Listen_ID  User_ID            Timestamp  Track_ID     Artist       Genre  \
0          1      167  2025-01-23 22:19:45       146  Artist_46  Electronic   
1          3       90  2025-01-06 15:32:21       105   Artist_5  Electronic   
2          5       39  2025-02-12 11:12:04       119  Artist_19     Hip-Hop   
3          6      126  2025-03-05 19:52:42       149  Artist_49         Pop   
4          8      173   2025-01-04 5:56:14       101   Artist_1         Pop   

   Duration_ms  Minutes_Played  
0       121062            2.02  
1       285983            4.77  
2       134397            2.24  
3       272617            4.54  
4       205999            3.43  


### Applying Functions: 
You write a custom function (or use a lambda) and apply it via .apply() to categorize the time of day into "Morning", "Afternoon", "Evening" and "Night" based on the Timestamp.

In [7]:
df_merge['Timestamp'] = pd.to_datetime(df_merge['Timestamp'])

def get_shift(ts):
    hour = ts.hour
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

df_merge['Shift'] = df_merge['Timestamp'].apply(get_shift)

print(df_merge[['Timestamp', 'Shift']].head())

            Timestamp      Shift
0 2025-01-23 22:19:45      Night
1 2025-01-06 15:32:21  Afternoon
2 2025-02-12 11:12:04    Morning
3 2025-03-05 19:52:42    Evening
4 2025-01-04 05:56:14    Morning


### Grouping and Aggregation:
Using .groupby(), you group the data by 'Genre' and use .agg() to find the total minutes played, the average song duration, and the unique count of listeners for each genre.

In [8]:
genre_metrics = df_merge.groupby('Genre').agg(
    Total_Minutes=('Minutes_Played', 'sum'),
    Avg_Duration_ms=('Duration_ms', 'mean'),
    Unique_Listeners=('User_ID', 'nunique')
).reset_index()

genre_metrics = genre_metrics.sort_values(by='Total_Minutes', ascending=False)
genre_metrics = genre_metrics.round(2)

print(genre_metrics)

        Genre  Total_Minutes  Avg_Duration_ms  Unique_Listeners
1  Electronic        1320.26        218211.60               164
5        Rock        1317.89        226570.80               170
2     Hip-Hop        1089.49        229298.52               154
4         Pop        1068.76        222772.09               162
3        Jazz         569.72        204591.54               117
0   Classical         215.92        270042.23                43


### Reshaping and Pivoting: 
Finally, you use pd.pivot_table() to create a heat-map-ready matrix showing "User Age Group" as rows, "Music Genre" as columns, and "Total Listens" as the values.

In [9]:
df_final = pd.merge(df_merge, df1, left_on='User_ID', right_on='user_id', how='left')

cols_to_drop = ['user_id']
df_final = df_final.drop(columns=cols_to_drop)

heatmap_data = pd.pivot_table(
    df_final, 
    values='Listen_ID',    
    index='User Age Group', 
    columns='Genre', 
    aggfunc='count',       
    fill_value=0
)

print(heatmap_data)

Genre           Classical  Electronic  Hip-Hop  Jazz  Pop  Rock
User Age Group                                                 
18-24                  11          69       65    27   66    72
25-34                   8          70       61    32   58    58
35-44                   8          59       54    32   58    72
45-54                  13          98       56    44   53    91
55+                     8          67       49    32   53    56


In [10]:
df_final

,Listen_ID,User_ID,Timestamp,Track_ID,Artist,Genre,Duration_ms,Minutes_Played,Shift,User Age Group,subscription_tier,sys_user_hash
0,1,167,2025-01-23 22:19:45,146,Artist_46,Electronic,121062,2.02,Night,45-54,Premium,usr_4734
1,3,90,2025-01-06 15:32:21,105,Artist_5,Electronic,285983,4.77,Afternoon,35-44,Free,usr_5324
2,5,39,2025-02-12 11:12:04,119,Artist_19,Hip-Hop,134397,2.24,Morning,18-24,Premium,usr_8634
3,6,126,2025-03-05 19:52:42,149,Artist_49,Pop,272617,4.54,Evening,45-54,Free,usr_3067
4,8,173,2025-01-04 05:56:14,101,Artist_1,Pop,205999,3.43,Morning,25-34,Free,usr_6438
...,...,...,...,...,...,...,...,...,...,...,...,...
1495,1494,175,2025-06-15 07:30:32,126,Artist_26,Rock,298352,4.97,Morning,45-54,Free,usr_6803
1496,1495,165,2025-05-22 13:18:12,101,Artist_1,Pop,205999,3.43,Afternoon,35-44,Free,usr_1403
1497,1496,49,2025-06-13 22:29:31,128,Artist_28,Classical,267851,4.46,Night,25-34,Free,usr_4520
1498,1497,120,2025-04-26 02:20:18,137,Artist_37,Hip-Hop,259407,4.32,Night,25-34,Premium,usr_1888
